# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vincentoei/flyrank-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item (page) as of the decision date.

Decision date: 2026-03-31.

Feature window: 2026-01-01 to 2026-03-31 (90 days before decision).

Target window: 2026-04-01 to 2026-04-30 (30 days after decision).

Tables: fact_content_daily_performance for daily performance, joined to dim_content for static page context.

What I predict: growth_label_30d = 1 if the page's impressions in 2026-04-01 to 2026-04-30 are at least 20% higher than in 2026-03-01 to 2026-03-31, and the prior-30-day window has at least 100 impressions.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import getpass
import duckdb

# Load token
token = os.getenv("HF_TOKEN")
if not token:
    token = getpass.getpass("Hugging Face token: ")

conn = duckdb.connect()

# Install and load HTTPFS
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")

# Create HTTP secret
conn.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{
            'Authorization': 'Bearer {token}'
        }}
    );
""")

# Attach fact table (all months, or restrict to one month for faster iteration)
conn.execute("""
    CREATE OR REPLACE VIEW fact AS
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
""")

# Attach dimension tables (single parquet files)
conn.execute("""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
""")

conn.execute("""
    CREATE OR REPLACE VIEW dim_clients AS
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')
""")

print("Warehouse views attached.")


Warehouse views attached.


In [6]:
result = conn.execute("""
    SELECT COUNT(*) AS row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result)

grain_check = conn.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(grain_check)
print(f"\nDuplicate rows found: {len(grain_check)}")

   row_count
0    9841378
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Duplicate rows found: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|-------|--------|-----|
| `content_hash_id`, `client_hash_id` | Context | Join keys and grouping only. Never used as model features. |
| `report_date` | Context | Used to define feature windows and joins. Not a model input. |
| `impressions` (summed over feature window) | Feature | Real Google Search Console (GSC) measurement available before the decision date. |
| `gsc_avg_position` (averaged over feature window) | Feature | Available before the decision date. |
| `sessions` (summed over feature window, where `ga4_data_available = TRUE`) | Feature | Real Google Analytics 4 (GA4) measurement available before the decision date. |
| `content_age_days` | Feature | Derived from `content_created_at` in `dim_content`; fixed at the decision time. |
| `growth_label_30d` | Label / Proxy | The future outcome being predicted. Computed using data from the future (e.g., 2026-04). |
| `days_since_last_update` | Excluded |  Warehouse record refresh date, not knowable at decision moment. |
| `trend_direction`, `trend_pct` | Excluded | Derived from the label/current target window, causing data leakage in the starter CSV. |
| `fact_content_query_90d` columns | Excluded | The 90-day query window overlaps with the prediction target window, creating leakage risk. |
| `health_score`, `priority_score`, `action_type` | Excluded | Product decision outputs. Although not present in the dataset, recreating them would introduce circular reasoning. |

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Print the contract buckets as a sanity check
contract = {
    "context": [
        "content_hash_id",
        "client_hash_id",
        "report_date",
    ],
    "features": [
        "impressions_90d",
        "gsc_avg_position_90d",
        "sessions_organic_90d",
        "content_age_days",
        "word_count",
    ],
    "label": ["growth_label_30d"], 
    "excluded": [
        "trend_direction",
        "trend_pct",
        "fact_content_query_90d columns",
        "health_score / priority_score / action_type",
    ],
}

for bucket, fields in contract.items():
    print(f"\n{bucket.upper()} ({len(fields)} fields):")
    for f in fields:
        print(f"  - {f}")


CONTEXT (3 fields):
  - content_hash_id
  - client_hash_id
  - report_date

FEATURES (5 fields):
  - impressions_90d
  - gsc_avg_position_90d
  - sessions_organic_90d
  - content_age_days
  - word_count

LABEL (1 fields):
  - growth_label_30d

EXCLUDED (4 fields):
  - trend_direction
  - trend_pct
  - fact_content_query_90d columns
  - health_score / priority_score / action_type


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3.1. Grain check

Query 1: verify the grain of fact_content_daily_performance for March 2026. One row per report_date + client_hash_id + content_hash_id.

### 3.2. Row count and date span

Query 2: row count and date span for the mid-panel month.

### 3.3. Availability check

Query 3: GA4 availability must be checked with IS TRUE, not = TRUE. Rows from before a client's GA4 start are zero-filled with ga4_data_available = FALSE, and millions more carry NULL — both must be excluded.

### 3.4. Five-feature frame

Build the five-feature frame from the feature window ending 2026-03-31. Every feature is knowable before the decision date. Feature availability notes:
- impressions_90d: knowable at decision moment because it sums only gsc_impressions from 2026-01-01 to 2026-03-31.
- avg_position_90d: knowable at decision moment because it averages gsc_avg_position over the same pre-decision window.
- sessions_organic_90d: knowable at decision moment because it sums sessions_organic from 2026-01-01 to 2026-03-31, filtered to rows where ga4_data_available IS TRUE.
- content_age_days: knowable at decision moment because content_created_date is fixed and we filter to pages created on or before 2026-03-31.
- word_count: knowable at decision moment because it is static content metadata in dim_content.

### 3.5. The trap - deliberate leakage

Deliberately add a label-derived feature: the impressions from the target window (April 2026). This should make the score near-perfect. Then remove it.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 3.1. Grain check
grain_check = conn.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print("Grain check: rows with duplicate grain (should be empty):")
print(grain_check)
print(f"\nRows returned: {len(grain_check)}")

# 3.2. Row count and date span
count_span = conn.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(count_span)

# 3.3. Availability check
availability = conn.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_not_available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(availability)
print(f"\nShare with GA4 available: {availability['ga4_available_rows'][0] / availability['total_rows'][0]:.3%}")

# 3.4. Five-feature frame
features = conn.execute("""
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            AVG(gsc_avg_position) AS avg_position_90d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    decision_date AS (
        SELECT DATE '2026-03-31' AS d
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.impressions_90d,
        f.avg_position_90d,
        f.sessions_organic_90d,
        (d.d - dim_content.content_created_date)::INTEGER AS content_age_days,
        dim_content.word_count
    FROM feature_window f
    LEFT JOIN dim_content
        ON f.content_hash_id = dim_content.content_hash_id
    CROSS JOIN decision_date d
    WHERE f.impressions_90d >= 100
      AND dim_content.content_created_date <= d.d
      AND dim_content.word_count IS NOT NULL
    LIMIT 5
""").df()

print("Five-feature frame preview:")
print(features)

# 3.5. The trap - deliberate leakage
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

honest = conn.execute("""
    WITH feature_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            AVG(gsc_avg_position) AS avg_position_90d,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_organic ELSE 0 END) AS sessions_organic_90d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-0[1-3]/*.parquet')
        WHERE report_date BETWEEN '2026-01-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    last_30 AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_last_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),
    label_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_next_30d
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
        WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
        GROUP BY client_hash_id, content_hash_id
    ),
    decision_date AS (
        SELECT DATE '2026-03-31' AS d
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.impressions_90d,
        f.avg_position_90d,
        f.sessions_organic_90d,
        l30.impressions_last_30d,
        l.impressions_next_30d,
        (d.d - dim_content.content_created_date)::INTEGER AS content_age_days,
        dim_content.word_count,
        CASE
            WHEN l.impressions_next_30d >= 1.20 * l30.impressions_last_30d
                 AND l30.impressions_last_30d >= 100
            THEN 1 ELSE 0
        END AS growth_label
    FROM feature_window f
    LEFT JOIN last_30 l30
        ON f.content_hash_id = l30.content_hash_id
    LEFT JOIN label_window l
        ON f.content_hash_id = l.content_hash_id
    LEFT JOIN dim_content
        ON f.content_hash_id = dim_content.content_hash_id
    CROSS JOIN decision_date d
    WHERE f.impressions_90d >= 300
      AND dim_content.content_created_date <= d.d
      AND dim_content.word_count IS NOT NULL
""").df()
    
honest_features = [
    "impressions_90d",
    "avg_position_90d",
    "sessions_organic_90d",
    "content_age_days",
    "word_count",
]

honest = honest.dropna(subset=honest_features + ["growth_label"])

# Honest model
X = honest[honest_features]
y = honest["growth_label"]
honest_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
honest_model.fit(X, y)
honest_auc = roc_auc_score(y, honest_model.predict_proba(X)[:, 1])
print(f"Honest AUC (5 features): {honest_auc:.3f}")

# Leaky model: add the future impressions
leaky_features = honest_features + ["impressions_next_30d"]
leaky_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
leaky_model.fit(honest[leaky_features], y)
leaky_auc = roc_auc_score(y, leaky_model.predict_proba(honest[leaky_features])[:, 1])
print(f"Leaky AUC (with future impressions): {leaky_auc:.3f}")

print("\nThe leaky feature is removed. The honest AUC is the number we keep.")

# The leaky feature impressions_next_30d pushes the AUC toward perfect. That is the leakage lesson. We keep only the five honest features.

Grain check: rows with duplicate grain (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []

Rows returned: 0
   row_count   min_date   max_date  client_count  content_count
0    9841378 2026-03-01 2026-03-31            55         331437
   total_rows  ga4_available_rows  ga4_not_available_rows
0     9841378            413966.0               9427412.0

Share with GA4 available: 4.206%
Five-feature frame preview:
            client_hash_id           content_hash_id  impressions_90d  \
0  client_3197e6291363b4db  content_ada8050325cb068f           2649.0   
1  client_08a6a72ff48e62c0  content_adbe651d0b6a0c25            631.0   
2  client_08a6a72ff48e62c0  content_98b8ac0868c87575            194.0   
3  client_08a6a72ff48e62c0  content_ab41edaa792cd770            274.0   
4  client_08a6a72ff48e62c0  content_8b82431361f7b6ac           2928.0   

   avg_position_90d  sessions_organic_90d  content_age_days  word_count  
0          8.183516 

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice has five hard limits:

1. Unbalanced panel. Clients joined the platform at different times, so some have no 2026-01 or 2026-02 history. A global 90-day feature window does not exist for everyone.

2. GSC-only early rows and sparse GA4. Only 4.2% of March 2026 rows have GA4 data available. sessions_organic_90d is mostly zero not because pages have no sessions, but because most clients did not share GA4 for that date or their GA4 start is later.

3. Freshness dates are not knowable at the decision point. content_updated_date and last_optimized_date contain values after 2026-03-31. They are warehouse or system timestamps, not content edit events. We cannot use them as pre-decision features.

4. Window overlap excludes the query table. fact_content_query_90d covers a fixed 90-day window that overlaps the target window (April 2026). Using it would leak the answer.

5. Future-created pages. Some dim_content rows have content_created_date after 2026-03-31. We must filter these out so the feature window actually exists.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Limitation 1: unbalanced panel — clients with the fewest March 2026 observation days
panel_limit = conn.execute("""
    SELECT
        client_hash_id,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS days_seen
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id
    ORDER BY days_seen
    LIMIT 10
""").df()

print("Clients with shortest March 2026 history:")
print(panel_limit)

# Limitation 3: content_updated_date is not a pre-decision feature
future_update = conn.execute("""
    SELECT
        COUNT(*) AS total_pages,
        SUM(CASE WHEN content_updated_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS updated_after_decision,
        SUM(CASE WHEN last_optimized_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS optimized_after_decision,
        SUM(CASE WHEN last_optimized_date IS NULL THEN 1 ELSE 0 END) AS optimized_null
    FROM dim_content
""").df()

print("\nFreshness dates are not usable at 2026-03-31:")
print(future_update)

# Limitation 5: future-created pages
future_created = conn.execute("""
    SELECT
        COUNT(*) AS total_pages,
        SUM(CASE WHEN content_created_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS created_after_decision
    FROM dim_content
""").df()

print("\nPages created after decision date:")
print(future_created)


# The code demonstrates three concrete limits:
# 1. Shortest-history clients — some clients may have only a few days in March 2026.
# 2. Freshness dates in the future — most content_updated_date values are after 2026-03-31, and last_optimized_date is mostly null.
# 3. Pages created after the decision date — some content was created after 2026-03-31, so a 90-day feature window cannot exist for them.
# These limits justify why:
# - sessions_organic_90d is mostly zero.
# - days_since_last_update was replaced with word_count.
# - I filter to content_created_date <= '2026-03-31'.

Clients with shortest March 2026 history:
            client_hash_id first_date  last_date  days_seen
0  client_e00b29e582949543 2026-03-23 2026-03-31          9
1  client_810019792c9b8efc 2026-03-20 2026-03-31         12
2  client_f6f0cdf26d03d7bd 2026-03-19 2026-03-31         13
3  client_86ebc2f12c01f586 2026-03-03 2026-03-31         29
4  client_cd12bcfd98942aa1 2026-03-01 2026-03-31         31
5  client_19b89ee4fe3db6da 2026-03-01 2026-03-31         31
6  client_4a18d1793d92fb84 2026-03-01 2026-03-31         31
7  client_8ae2bfb5aa1ffa1e 2026-03-01 2026-03-31         31
8  client_ba65e80a1116ae41 2026-03-01 2026-03-31         31
9  client_c182d11e4862a37d 2026-03-01 2026-03-31         31

Freshness dates are not usable at 2026-03-31:
   total_pages  updated_after_decision  optimized_after_decision  \
0       519606                382739.0                   45396.0   

   optimized_null  
0        474210.0  

Pages created after decision date:
   total_pages  created_after_decision

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.